# Modelado — validación temporal de matrices compactas

Orquesta `src.modeling.validacion_temporal_compacta`. Compara tres matrices predefinidas con los hiperparámetros fijos. El umbral de clasificación de cada temporada usa solo predicciones previas cuyas etiquetas estaban cerradas al primer origen. La selección usa 2022–2023; 2024 es un contraste ya inspeccionado y 2025 una sensibilidad con tres positivos.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modeling.validacion_temporal_compacta import ejecutar_validacion_temporal_compacta

reporte = ejecutar_validacion_temporal_compacta()

h=2: 30160 filas comunes; calibración progresiva
h=4: 30030 filas comunes; calibración progresiva
MLflow: run 24c539d6256b425f82206a28836d5bd3 en 'denguekay/validacion_temporal_compacta' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


In [2]:
import pandas as pd

filas = []
for h, bloque in reporte["horizontes"].items():
    for variante, resultado in bloque["variantes"].items():
        seleccion = resultado["folds"][:2]
        contraste = resultado["folds"][2]
        for enfoque, ruta in (("regresion", "alerta_derivada"), ("clasificacion", "alerta_directa")):
            filas.append({
                "h": int(h), "objetivo": enfoque, "variante": variante,
                "F1_2022_2023": sum(f[enfoque][ruta]["f1"] for f in seleccion) / 2,
                "F1_2024": contraste[enfoque][ruta]["f1"],
                "seleccionada": bloque["seleccion_2022_2023"][enfoque] == variante,
            })
pd.DataFrame(filas).sort_values(["h", "objetivo", "variante"]).round(3)

,h,objetivo,variante,F1_2022_2023,F1_2024,seleccionada
1,2,clasificacion,base_4,0.668,0.650,False
3,2,clasificacion,base_6,0.672,0.636,False
5,2,clasificacion,base_6_poblacion_2017,0.696,0.643,True
0,2,regresion,base_4,0.751,0.799,False
2,2,regresion,base_6,0.754,0.805,True
4,2,regresion,base_6_poblacion_2017,0.741,0.801,False
7,4,clasificacion,base_4,0.610,0.609,False
9,4,clasificacion,base_6,0.608,0.614,False
11,4,clasificacion,base_6_poblacion_2017,0.663,0.623,True
6,4,regresion,base_4,0.694,0.757,True


La interpretación, la persistencia y los límites están en `docs/modeling/validacion_temporal_compacta.md`. El JSON conserva umbrales, cortes, calibraciones y métricas por temporada.